# Hotel Bar Inventory Forecasting and Par Level Recommendations

This notebook turns raw inventory movements into daily bar-brand demand, selects a transparent demand model per series, calculates dynamic par levels, and backtests a daily-review replenishment policy.

**Key assumptions:** 2-day fixed lead time, 95% cycle service level (Z=1.645), daily review, 30 ml ordering increment, and unmet demand is lost. Demand in historical stockout periods may be censored, so results should be treated as conservative.

In [11]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

RAW_PATH = Path(r'C:\Users\anand\Downloads\Consumption Dataset.xlsx')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
LEAD_TIME_DAYS, SERVICE_Z, ROUNDING_ML = 2, 1.645, 30

def wape(actual, forecast):
    actual, forecast = np.asarray(actual), np.asarray(forecast)
    return np.abs(actual - forecast).sum() / actual.sum() if actual.sum() else np.nan

def ma7(history, date):
    return float(np.mean(history[-7:])) if history else 0.0

def seasonal_dow(history, date):
    matching = [history[-lag] for lag in range(7, min(len(history), 56)+1, 7)]
    return float(np.mean(matching)) if matching else ma7(history, date)

In [12]:
# Load, validate, aggregate, and fill zero-demand days
raw = pd.read_excel(RAW_PATH)
raw['Date Time Served'] = pd.to_datetime(raw['Date Time Served'], errors='coerce')
raw = raw.dropna(subset=['Date Time Served', 'Bar Name', 'Brand Name']).copy()
num = ['Opening Balance (ml)', 'Purchase (ml)', 'Consumed (ml)', 'Closing Balance (ml)']
raw[num] = raw[num].apply(pd.to_numeric, errors='coerce').fillna(0.0)
raw['Date'] = raw['Date Time Served'].dt.normalize()
raw['conservation_error_ml'] = raw['Closing Balance (ml)'] - (raw['Opening Balance (ml)'] + raw['Purchase (ml)'] - raw['Consumed (ml)'])
raw['conservation_pass'] = raw['conservation_error_ml'].abs() <= 0.02
print(f'Rows: {len(raw):,}; conservation pass rate: {raw.conservation_pass.mean():.1%}')
observed = raw.groupby(['Date','Bar Name','Brand Name'], as_index=False).agg(consumed_ml=('Consumed (ml)','sum'), alcohol_type=('Alcohol Type','first'), stockout_proxy=('Closing Balance (ml)', lambda x: (x <= .02).any()))
pairs = observed[['Bar Name','Brand Name','alcohol_type']].drop_duplicates(); dates = pd.date_range(observed.Date.min(), observed.Date.max(), freq='D')
grid = pairs.assign(_k=1).merge(pd.DataFrame({'Date':dates, '_k':1}), on='_k').drop(columns='_k')
daily = grid.merge(observed, on=['Date','Bar Name','Brand Name','alcohol_type'], how='left').fillna({'consumed_ml':0, 'stockout_proxy':False})
daily = daily.sort_values(['Bar Name','Brand Name','Date']).reset_index(drop=True)
daily.to_csv(OUTPUT_DIR/'daily_bar_consumption.csv', index=False)
daily.head()

Rows: 6,575; conservation pass rate: 100.0%


,Bar Name,Brand Name,alcohol_type,Date,consumed_ml,stockout_proxy
0,Anderson's Bar,Absolut,Vodka,2023-01-01,0.00,False
1,Anderson's Bar,Absolut,Vodka,2023-01-02,118.62,False
2,Anderson's Bar,Absolut,Vodka,2023-01-03,0.00,False
3,Anderson's Bar,Absolut,Vodka,2023-01-04,0.00,False
4,Anderson's Bar,Absolut,Vodka,2023-01-05,463.20,False


In [13]:
# EDA: velocity tiers and weekday demand
velocity = daily.groupby(['Bar Name','Brand Name'], as_index=False).agg(total_consumption_ml=('consumed_ml','sum')).sort_values('total_consumption_ml', ascending=False)
velocity['cumulative_share'] = velocity.total_consumption_ml.cumsum()/velocity.total_consumption_ml.sum()
velocity['ABC Class'] = np.select([velocity.cumulative_share <= .80, velocity.cumulative_share <= .95], ['A','B'], default='C')
print(velocity.groupby('ABC Class').agg(series=('Brand Name','size'), consumption_ml=('total_consumption_ml','sum')))
weekday = daily.assign(day_of_week=daily.Date.dt.day_name()).groupby('day_of_week').consumed_ml.mean()
weekday.reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])

           series  consumption_ml
ABC Class                        
A              69      1559483.07
B              19       308240.44
C               8       100958.15


day_of_week
Monday       53.224733
Tuesday      56.017458
Wednesday    58.503101
Thursday     55.643606
Friday       56.826180
Saturday     56.474858
Sunday       55.584892
Name: consumed_ml, dtype: float64

In [14]:
# Per-series model selection: choose the lowest causal WAPE on the training period.
# Candidates are a 7-day moving average and a same-weekday seasonal mean.
# The last 20% of days remain untouched for final backtesting.
forecast_rows, model_rows = [], []
for (bar, brand), g in daily.groupby(['Bar Name','Brand Name']):
    g = g.sort_values('Date').reset_index(drop=True); y = g.consumed_ml.to_numpy(float); ds = g.Date.tolist(); split = max(28, math.ceil(.8*len(g)))
    funcs = {'7-day moving average':ma7, 'day-of-week seasonal mean':seasonal_dow}
    scores = {name:wape(y[28:split], [fn(y[:i].tolist(), ds[i]) for i in range(28,split)]) for name,fn in funcs.items()}
    chosen = min(scores, key=lambda k: np.inf if np.isnan(scores[k]) else scores[k]); fn = funcs[chosen]
    train_pred = np.array([fn(y[:i].tolist(), ds[i]) for i in range(28,split)]); rmse = float(np.sqrt(np.mean((y[28:split]-train_pred)**2)))
    pred = np.array([fn(y[:i].tolist(), ds[i]) for i in range(split,len(g))]); actual = y[split:]
    model_rows.append({'Bar Name':bar,'Brand Name':brand,'Chosen Model':chosen,'Validation WAPE':scores[chosen],'Test WAPE':wape(actual,pred),'RMSE (ml)':rmse})
    for date,a,p in zip(ds[split:],actual,pred):
        par = math.ceil((p*LEAD_TIME_DAYS + SERVICE_Z*rmse*np.sqrt(LEAD_TIME_DAYS))/ROUNDING_ML)*ROUNDING_ML
        forecast_rows.append({'Date':date,'Bar Name':bar,'Brand Name':brand,'Actual Demand (ml)':a,'Forecast (ml)':p,'Par Level (ml)':par,'Chosen Model':chosen})
models = pd.DataFrame(model_rows); forecast = pd.DataFrame(forecast_rows)
print('Final holdout WAPE:', wape(forecast['Actual Demand (ml)'], forecast['Forecast (ml)']))
models.head()

Final holdout WAPE: 1.6929999688231765


,Bar Name,Brand Name,Chosen Model,Validation WAPE,Test WAPE,RMSE (ml)
0,Anderson's Bar,Absolut,7-day moving average,1.650200,1.754729,98.807633
1,Anderson's Bar,Bacardi,7-day moving average,1.604829,1.434381,174.382612
2,Anderson's Bar,Barefoot,day-of-week seasonal mean,1.496434,2.302135,171.825928
3,Anderson's Bar,Budweiser,day-of-week seasonal mean,1.727622,NaN,117.077546
4,Anderson's Bar,Captain Morgan,day-of-week seasonal mean,1.553679,1.798176,181.733603


In [15]:
# Simulate daily review / order-up-to-par policy on the holdout.
# Orders arrive two days after placement; inventory position includes on-order stock.
sim_rows = []
for (bar, brand), g in forecast.groupby(['Bar Name','Brand Name']):
    g = g.sort_values('Date'); stock = float(g.iloc[0]['Par Level (ml)']); pending=[]
    for r in g.itertuples(index=False):
        arriving = sum(q for due,q in pending if due == r.Date); stock += arriving; pending=[(due,q) for due,q in pending if due != r.Date]
        fulfilled=min(stock, r._3); lost=r._3-fulfilled; stock-=fulfilled
        position=stock+sum(q for _,q in pending); order=max(0.0, r._5-position)
        if order: pending.append((r.Date+pd.Timedelta(days=LEAD_TIME_DAYS), order))
        sim_rows.append([r.Date,bar,brand,r._3,r._4,r._5,stock,order,lost,lost>0])
simulation=pd.DataFrame(sim_rows, columns=['Date','Bar Name','Brand Name','Demand (ml)','Forecast (ml)','Par Level (ml)','Closing Inventory (ml)','Order Quantity (ml)','Lost Demand (ml)','Stockout Day'])
print(simulation.agg({'Stockout Day':'sum','Lost Demand (ml)':'sum','Closing Inventory (ml)':'mean'}))
forecast.to_csv(OUTPUT_DIR/'forecast_recommendations.csv', index=False); simulation.to_csv(OUTPUT_DIR/'simulation_results.csv', index=False); models.to_csv(OUTPUT_DIR/'model_performance_by_series.csv', index=False)

Stockout Day                255.000000
Lost Demand (ml)          42874.310000
Closing Inventory (ml)      467.161336
dtype: float64


## Operating use
Each morning, refresh the source file, re-run the notebook, and publish `forecast_recommendations.csv`. A manager orders the difference between par level and inventory position (on-hand plus confirmed inbound). Investigate any item with repeated stockout proxy days, large forecast error, missing transactions, promotions, or supplier delays before auto-ordering.